Aumentar la  de clasificación binaria y la cantidad de iteraciones para ver hasta qué punto se puede mejorar el puntaje F1, especialmente en el set de validación y pruebas. Reportar cuál de los sets de datos es el que permite el mejor desempeño luego de utilizar un modelo más complejo. Seguidamente, responda:

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd

# Cargar el dataset
df = pd.read_csv("diabetes.csv", skiprows=1)  # Ignora la fila con el texto
df.columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
              'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']


# Separación de datos
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
# Modelo base de regresión logística
modelo_base = LogisticRegression(max_iter=1000)
modelo_base.fit(X_train, y_train)

# Evaluación con F1
y_pred_val_base = modelo_base.predict(X_val)
f1_base = f1_score(y_val, y_pred_val_base)

print("F1 del modelo base:", f1_base)
print(classification_report(y_val, y_pred_val_base))

F1 del modelo base: 0.6548672566371682
              precision    recall  f1-score   support

           0       0.81      0.79      0.80        99
           1       0.64      0.67      0.65        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154



In [22]:
# Modelo con mayor complejidad
modelo_complejo = LogisticRegression(
    max_iter=50000,  # muchas más iteraciones
    C=50,           # menor regularización => modelo más complejo
    solver='lbfgs'
)

# Entrenamiento del modelo complejo
modelo_complejo.fit(X_train, y_train)

# Evaluación con F1
y_pred_val_complejo = modelo_complejo.predict(X_val)
f1_complejo = f1_score(y_val, y_pred_val_complejo)

print("F1 del modelo complejo:", f1_complejo)
print(classification_report(y_val, y_pred_val_complejo))

F1 del modelo complejo: 0.6607142857142857
              precision    recall  f1-score   support

           0       0.81      0.80      0.81        99
           1       0.65      0.67      0.66        55

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.73       154
weighted avg       0.76      0.75      0.75       154



1. De acuerdo con los resultados base obtenidos con la regresión logística original, el puntaje F1 aumenta en el set de validación al rellenar valores inválidos y eliminar los outliers, ¿esto mismo se cumple al aumentar la complejidad del modelo?

El modelo complejo incrementa ligeramente el F1 score, lo que significa que aumentar la complejidad sí produjo una pequeña mejora en la capacidad del modelo para identificar correctamente a los diabéticos.

Sin embargo, la mejora es mínima, lo que sugiere que el modelo original ya captura bien la estructura de los datos. Incrementar la complejidad no garantizó una mejora sustancial, posible límite de la información disponible.

2. Si quisiéramos optimizar el modelo para reducir al máximo la cantidad de falsos negativos (personas con diabetes clasificadas como no diabéticas), ¿cuál sería la métrica más conveniente? Realice las modificaciones a nivel de código para mostrar el resultado de esta métrica.

Para este caso, lo más importante es no dejar pasar a alguien enfermo como sano, por lo tanto, se debe maximizar el recall o sensibilidad de la clase positiva.

In [23]:
# Modelo orientado a maximizar recall
modelo_recall = LogisticRegression(
    max_iter=50000,
    C=50,
    class_weight={0:1, 1:3},  # penaliza más los errores en la clase 1
    solver='lbfgs'
)

modelo_recall.fit(X_train, y_train)

# Predicción
y_pred_val = modelo_recall.predict(X_val)

# Métrica principal: Recall clase positiva (diabéticos)
recall_diabeticos = recall_score(y_val, y_pred_val, pos_label=1)

print("Recall (clase diabéticos):", recall_diabeticos)
print(classification_report(y_val, y_pred_val))


Recall (clase diabéticos): 0.8181818181818182
              precision    recall  f1-score   support

           0       0.86      0.62      0.72        99
           1       0.54      0.82      0.65        55

    accuracy                           0.69       154
   macro avg       0.70      0.72      0.68       154
weighted avg       0.75      0.69      0.69       154

